In [1]:
from incidentiq.search import SearchEngine
DATA_PATH = r"E:\incidentiq\data\processed\logs.parquet"
engine = SearchEngine(DATA_PATH)

e:\incidentiq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 63/63 [00:04<00:00, 14.44it/s]


In [2]:
RELEVANCE_STRONG = 2
RELEVANCE_SOMEWHAT = 1
RELEVANCE_IRRELEVANT = 0

In [3]:
qrels = {
    "hardware stopped working": {

        # =========================
        # STRONGLY RELEVANT
        # =========================

        1329: 2,
        1216: 2,
        1219: 2,
        1230: 2,
        1407: 2,

        1948: 2,
        1218: 2,
        457: 2,
        1227: 2,
        1229: 2,
        620: 2,

        450: 2,

        231: 2,
        232: 2,
        233: 2,
        234: 2,
        238: 2,

        1029: 2,


        # =========================
        # SOMEWHAT RELEVANT
        # =========================

        1406: 1,
        521: 1,
        1221: 1,
        1224: 1,
        1220: 1,

        1739: 1,
        1747: 1,
        1756: 1,
        1259: 1,
        1519: 1,

        1258: 1,
        1202: 1,

        1985: 1,
        1982: 1,
        1962: 1,
        1745: 1,
        1480: 1,
        1459: 1,
        1483: 1,
        1373: 1,
        1495: 1,

        1281: 1,
        1069: 1,
        1456: 1,
        314: 1,
        837: 1,
        1234: 1,
        1514: 1,
        1261: 1,
        1402: 1,


        # =========================
        # IRRELEVANT
        # =========================

        1370: 0,
        1371: 0,
    }
}

In [4]:
judgments = qrels["hardware stopped working"]

print("Total judgments:", len(judgments))

print(
    "Strong:",
    sum(score == 2 for score in judgments.values())
)

print(
    "Somewhat:",
    sum(score == 1 for score in judgments.values())
)

print(
    "Irrelevant:",
    sum(score == 0 for score in judgments.values())
)

Total judgments: 50
Strong: 18
Somewhat: 30
Irrelevant: 2


In [5]:
for doc_id, relevance in judgments.items():

    print(
        f"doc={doc_id:4} | relevance={relevance}"
    )

doc=1329 | relevance=2
doc=1216 | relevance=2
doc=1219 | relevance=2
doc=1230 | relevance=2
doc=1407 | relevance=2
doc=1948 | relevance=2
doc=1218 | relevance=2
doc= 457 | relevance=2
doc=1227 | relevance=2
doc=1229 | relevance=2
doc= 620 | relevance=2
doc= 450 | relevance=2
doc= 231 | relevance=2
doc= 232 | relevance=2
doc= 233 | relevance=2
doc= 234 | relevance=2
doc= 238 | relevance=2
doc=1029 | relevance=2
doc=1406 | relevance=1
doc= 521 | relevance=1
doc=1221 | relevance=1
doc=1224 | relevance=1
doc=1220 | relevance=1
doc=1739 | relevance=1
doc=1747 | relevance=1
doc=1756 | relevance=1
doc=1259 | relevance=1
doc=1519 | relevance=1
doc=1258 | relevance=1
doc=1202 | relevance=1
doc=1985 | relevance=1
doc=1982 | relevance=1
doc=1962 | relevance=1
doc=1745 | relevance=1
doc=1480 | relevance=1
doc=1459 | relevance=1
doc=1483 | relevance=1
doc=1373 | relevance=1
doc=1495 | relevance=1
doc=1281 | relevance=1
doc=1069 | relevance=1
doc=1456 | relevance=1
doc= 314 | relevance=1
doc= 837 | 

In [6]:
def get_relevance_list(results, qrels):

    return [
        qrels.get(
            result["doc_id"],
            0
        )
        for result in results
    ]

In [7]:
query = "hardware stopped working"

semantic_results = engine.search_semantic(
    query,
    top_k=10
)

relevance = get_relevance_list(
    semantic_results,
    qrels[query]
)

print(relevance)

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [8]:
def precision_at_k(results, qrels, k):
    top_results = results[:k]

    relevant = sum(
        qrels.get(result["doc_id"], 0) > 0
        for result in top_results
    )

    return relevant / k

In [9]:
precision = precision_at_k(
    semantic_results,
    qrels[query],
    k=10
)

print(f"Precision@10: {precision:.2f}")

Precision@10: 1.00


In [10]:
query = "hardware stopped working"

bm25_results = engine.search_bm25(
    query,
    top_k=10
)

semantic_results = engine.search_semantic(
    query,
    top_k=10
)

rrf_results = engine.search_hybrid(
    query,
    top_k=10
)

In [11]:
systems = {
    "BM25": bm25_results,
    "Semantic": semantic_results,
    "RRF": rrf_results,
}

for name, results in systems.items():

    precision = precision_at_k(
        results,
        qrels[query],
        k=10
    )

    print(
        f"{name:10} | Precision@10 = {precision:.2f}"
    )

BM25       | Precision@10 = 0.00
Semantic   | Precision@10 = 1.00
RRF        | Precision@10 = 1.00


In [ ]:
import math


def dcg_at_k(relevance_scores, k):
    relevance_scores = relevance_scores[:k]

    dcg = 0.0

    for rank, relevance in enumerate(
        relevance_scores,
        start=1
    ):
        dcg += (
            (2 ** relevance - 1)
            / math.log2(rank + 1)
        )

    return dcg